# 05 · 特征清洗与时间外验证划分

本阶段承接 Notebook 4 的申请级特征宽表，构建固定的时间切分和可复用的清洗配置。

**处理范围：**

1. 固定 Train / Validation / OOT 的时间边界；
2. 只使用 Train 确定基础清洗规则；
3. 删除 Train 中全空或常量的特征；
4. 将历史日期转换成相对申请日的天数；
5. 将同一套规则应用到 Validation、OOT 和官方 Test；
6. 保存后续 Notebook 需要的数据、字段目录和配置。

**范围限制：** 暂不处理 near-constant、复杂缺失模式、来源表存在标记和预测时可用性审计；这些仍需要在后续阶段单独验证。

## 数据流

```text
Notebook 4 输出的申请级宽表
        ↓
读取关键字段并确认时间范围
        ↓
Train: 0–63 周
Validation: 64–77 周
OOT: 78–91 周
        ↓
只在 Train 上确定：
全空列 / 常量列 / 日期转换规则
        ↓
同一套规则应用到所有数据
        ↓
输出 clean_split + sealed OOT target
```


## 1. 数据接力：接入上游宽表与运行环境

Notebook 4 已经生成两张最终宽表：

- `home_credit_features_train.parquet`
- `home_credit_features_test.parquet`

本 Notebook 继续沿用原项目的目录约定：

- `/kaggle/input`：读取 Notebook 4 的输出；
- `/kaggle/working/clean_split`：保存 Train、Validation、OOT features 和 Test；
- `/kaggle/working/sealed_labels`：单独保存 OOT target；
- `/kaggle/working/reports/feature_cleaning`：保存本轮检查结果。

数据量较大，因此主体逻辑使用 Pandas，Parquet 的 schema 和分批写出使用 PyArrow。PyArrow 在这里主要负责文件读写，不承担清洗逻辑。


In [1]:
from pathlib import Path
import gc
import json

import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
from IPython.display import display

INPUT_ROOT = Path("/kaggle/input")
OUTPUT_ROOT = Path("/kaggle/working/clean_split")
SEALED_ROOT = Path("/kaggle/working/sealed_labels")
REPORT_ROOT = Path("/kaggle/working/reports/feature_cleaning")

OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
SEALED_ROOT.mkdir(parents=True, exist_ok=True)
REPORT_ROOT.mkdir(parents=True, exist_ok=True)

TRAIN_FILENAME = "home_credit_features_train.parquet"
TEST_FILENAME = "home_credit_features_test.parquet"

ID_COL = "case_id"
TARGET_COL = "target"
WEEK_COL = "WEEK_NUM"
DECISION_DATE_COL = "date_decision"

# 项目固定时间边界
VALID_START_WEEK = 64
OOT_START_WEEK = 78
EXPECTED_WEEK_MIN = 0
EXPECTED_WEEK_MAX = 91

# 宽表分批处理参数
PROFILE_COLUMN_BATCH_SIZE = 32
WRITE_ROW_BATCH_SIZE = 10_000

SAVE_OUTPUTS = True

print("Pandas:", pd.__version__)
print("PyArrow:", pa.__version__)
print("输出目录:", OUTPUT_ROOT)


Pandas: 2.3.3
PyArrow: 24.0.0
输出目录: /kaggle/working/clean_split


## 2. 起点确认：核对 Train / Test 数据结构

首先验证上游特征宽表的输入结构。

Train 应包含 `target`，Test 不应包含 `target`；除此之外，两者的字段名、顺序和类型应保持一致。

同时检查四个关键字段：

- `case_id`：申请主键；
- `target`：训练标签；
- `WEEK_NUM`：时间切分依据；
- `date_decision`：申请日期，也是日期特征转换的基准。

这里只做结构检查，还不进行任何清洗。


In [2]:
def locate_unique(filename):
    matches = list(INPUT_ROOT.rglob(filename))

    if len(matches) != 1:
        raise FileNotFoundError(
            f"{filename} 应该只找到 1 份，实际找到 {len(matches)} 份：{matches}"
        )

    return matches[0]


def parquet_schema(path):
    return pq.ParquetFile(path).schema_arrow


def schema_signature(schema):
    return [(field.name, str(field.type)) for field in schema]


TRAIN_PATH = locate_unique(TRAIN_FILENAME)
TEST_PATH = locate_unique(TEST_FILENAME)

train_schema = parquet_schema(TRAIN_PATH)
test_schema = parquet_schema(TEST_PATH)

train_without_target = pa.schema([
    field
    for field in train_schema
    if field.name != TARGET_COL
])

assert schema_signature(train_without_target) == schema_signature(test_schema), (
    "Train/Test 的字段名、顺序或类型不一致"
)

assert TARGET_COL in train_schema.names
assert TARGET_COL not in test_schema.names

required_columns = {
    ID_COL,
    TARGET_COL,
    WEEK_COL,
    DECISION_DATE_COL,
}
missing_required = required_columns - set(train_schema.names)

assert not missing_required, (
    f"Train 缺少关键字段：{sorted(missing_required)}"
)

print("Train:", TRAIN_PATH)
print("Test :", TEST_PATH)
print(f"Train 字段数：{len(train_schema):,}")


Train: /kaggle/input/notebooks/jing78y38/4-zjing/joined/home_credit_features_train.parquet
Test : /kaggle/input/notebooks/jing78y38/4-zjing/joined/home_credit_features_test.parquet
Train 字段数：1,457


## 3. 时间划界：建立 Train / Validation / OOT

考虑到信用风险分布可能随时间变化，数据集按照 `WEEK_NUM` 保持时间顺序，而非随机切分。

固定边界为：

| 数据集 | WEEK_NUM | 用途 |
|---|---:|---|
| Train | 0–63 | 确定清洗规则并训练模型 |
| Validation | 64–77 | 模型选择和调参 |
| OOT | 78–91 | 最后模拟更未来的数据 |

时间边界在读取 `target` 之前就已经固定。这样后续不会因为看到某一段坏账率较高或较低，再反过来调整切分位置。

这一部分只读取 `case_id`、`WEEK_NUM` 和 `date_decision`。


In [3]:
time_data = pd.read_parquet(
    TRAIN_PATH,
    columns=[ID_COL, WEEK_COL, DECISION_DATE_COL],
    engine="pyarrow",
)

time_data[DECISION_DATE_COL] = pd.to_datetime(
    time_data[DECISION_DATE_COL],
    errors="coerce",
)

assert time_data[ID_COL].is_unique, f"{ID_COL} 不唯一"
assert time_data[WEEK_COL].notna().all(), f"{WEEK_COL} 存在缺失"

actual_week_min = int(time_data[WEEK_COL].min())
actual_week_max = int(time_data[WEEK_COL].max())

assert (actual_week_min, actual_week_max) == (
    EXPECTED_WEEK_MIN,
    EXPECTED_WEEK_MAX,
), (
    f"WEEK_NUM 实际范围为 {actual_week_min}–{actual_week_max}，"
    "与项目设定不一致"
)

split_masks = {
    "train": time_data[WEEK_COL] < VALID_START_WEEK,
    "validation": (
        (time_data[WEEK_COL] >= VALID_START_WEEK)
        & (time_data[WEEK_COL] < OOT_START_WEEK)
    ),
    "oot": time_data[WEEK_COL] >= OOT_START_WEEK,
}

split_labels = pd.Series(
    pd.NA,
    index=time_data.index,
    dtype="string",
)

for split_name, mask in split_masks.items():
    split_labels.loc[mask] = split_name

assert split_labels.notna().all()

split_mapping = time_data[
    [ID_COL, WEEK_COL, DECISION_DATE_COL]
].copy()

split_mapping["split"] = split_labels

SPLIT_MAPPING_PATH = OUTPUT_ROOT / "split_mapping.parquet"
split_mapping.to_parquet(
    SPLIT_MAPPING_PATH,
    index=False,
    engine="pyarrow",
)

split_summary_rows = []

for split_name, mask in split_masks.items():
    part = time_data.loc[mask]

    split_summary_rows.append({
        "split": split_name,
        "row_count": len(part),
        "unique_case_count": part[ID_COL].nunique(),
        "week_count": part[WEEK_COL].nunique(),
        "week_min": int(part[WEEK_COL].min()),
        "week_max": int(part[WEEK_COL].max()),
        "decision_date_min": part[DECISION_DATE_COL].min(),
        "decision_date_max": part[DECISION_DATE_COL].max(),
    })

split_summary = pd.DataFrame(split_summary_rows)

assert split_summary["row_count"].gt(0).all()
assert split_summary["row_count"].eq(
    split_summary["unique_case_count"]
).all()

split_summary.to_csv(
    REPORT_ROOT / "time_split_summary.csv",
    index=False,
)

display(split_summary)

print(
    f"固定边界：Train 0–{VALID_START_WEEK - 1}；"
    f"Validation {VALID_START_WEEK}–{OOT_START_WEEK - 1}；"
    f"OOT {OOT_START_WEEK}–{EXPECTED_WEEK_MAX}"
)


,split,row_count,unique_case_count,week_count,week_min,week_max,decision_date_min,decision_date_max
0,train,1282082,1282082,64,0,63,2019-01-01,2020-03-23
1,validation,92870,92870,14,64,77,2020-03-24,2020-06-29
2,oot,151707,151707,14,78,91,2020-06-30,2020-10-05


固定边界：Train 0–63；Validation 64–77；OOT 78–91


## 4. 字段分工：明确哪些信息进入模型

宽表中的字段用途不同，因此在清洗前先区分：

- `case_id`：主键，不作为模型特征；
- `target`：标签；
- `WEEK_NUM`、`date_decision`、`MONTH`：时间信息，保留给切分和后续按时间评价；
- 其余字段：模型候选特征。

字段类型同时分为：

- `numeric`
- `categorical`
- `boolean`
- `date`
- `other`

这个字段目录会在最后保存为 `feature_catalog.csv`，供后续 EDA 和模型 Notebook 继续使用。


In [4]:
def dtype_family(arrow_type):
    if pa.types.is_date(arrow_type) or pa.types.is_timestamp(arrow_type):
        return "date"

    if (
        pa.types.is_integer(arrow_type)
        or pa.types.is_floating(arrow_type)
        or pa.types.is_decimal(arrow_type)
    ):
        return "numeric"

    if pa.types.is_boolean(arrow_type):
        return "boolean"

    if (
        pa.types.is_string(arrow_type)
        or pa.types.is_large_string(arrow_type)
        or pa.types.is_dictionary(arrow_type)
    ):
        return "categorical"

    return "other"


TIME_METADATA_COLS = [
    column
    for column in [WEEK_COL, DECISION_DATE_COL, "MONTH"]
    if column in train_schema.names
]

MODEL_EXCLUDE_COLS = {
    ID_COL,
    TARGET_COL,
    *TIME_METADATA_COLS,
}

model_candidate_cols = [
    field.name
    for field in train_schema
    if field.name not in MODEL_EXCLUDE_COLS
]

print(f"模型候选特征数：{len(model_candidate_cols):,}")


模型候选特征数：1,452


## 5. 清洗取舍：只从 Train 确定删除规则

这一轮只处理最明确、最不需要主观判断的两类特征：

### 5.1 全空特征

如果一个字段在 Train 中完全没有有效值，模型无法从这个字段学习任何关系，因此删除。

### 5.2 常量特征

如果一个字段在 Train 的非缺失部分始终只有一个值，它同样无法区分不同样本，因此删除。

### 5.3 高缺失字段

高缺失本身不等于无用。某个字段即使缺失很多，也可能在少量非缺失样本中包含有效风险信息。因此这里只记录缺失率，不根据缺失率自动删除。

清洗规则只从 Train（0–63 周）产生。Validation 和 OOT 不参与决定哪些列被删除。

由于宽表字段很多，这一步按列分批读取；每一批的统计仍然使用完整 Train 行，而不是抽样。


In [5]:
train_row_count = int(
    split_summary.loc[
        split_summary["split"].eq("train"),
        "row_count",
    ].iloc[0]
)

profile_rows = []

for start in range(
    0,
    len(model_candidate_cols),
    PROFILE_COLUMN_BATCH_SIZE,
):
    batch = model_candidate_cols[
        start:start + PROFILE_COLUMN_BATCH_SIZE
    ]

    frame = pd.read_parquet(
        TRAIN_PATH,
        columns=[WEEK_COL, *batch],
        engine="pyarrow",
    )

    train_part = frame.loc[
        frame[WEEK_COL] < VALID_START_WEEK,
        batch,
    ]

    for column in batch:
        series = train_part[column]

        valid_count = int(series.notna().sum())
        non_missing_nunique = int(series.dropna().nunique())

        profile_rows.append({
            "column": column,
            "valid_count": valid_count,
            "missing_count": train_row_count - valid_count,
            "missing_rate": (
                (train_row_count - valid_count) / train_row_count
            ),
            "non_missing_nunique": non_missing_nunique,
            "is_all_missing": valid_count == 0,
            "is_constant": (
                valid_count > 0
                and non_missing_nunique <= 1
            ),
        })

    del frame, train_part
    gc.collect()

    finished = min(
        start + len(batch),
        len(model_candidate_cols),
    )
    print(
        f"已检查 {finished:,} / "
        f"{len(model_candidate_cols):,} 个候选特征"
    )

feature_profile = pd.DataFrame(profile_rows)

all_missing_cols = feature_profile.loc[
    feature_profile["is_all_missing"],
    "column",
].tolist()

constant_cols = feature_profile.loc[
    feature_profile["is_constant"],
    "column",
].tolist()

drop_cols = sorted(
    set(all_missing_cols + constant_cols)
)

feature_profile.to_csv(
    REPORT_ROOT / "train_feature_quality_basic.csv",
    index=False,
)

print(f"Train 全空特征：{len(all_missing_cols):,}")
print(f"Train 常量特征：{len(constant_cols):,}")
print(f"本轮删除特征：{len(drop_cols):,}")

display(
    feature_profile
    .sort_values("missing_rate", ascending=False)
    .head(20)
)


已检查 32 / 1,452 个候选特征
已检查 64 / 1,452 个候选特征
已检查 96 / 1,452 个候选特征
已检查 128 / 1,452 个候选特征
已检查 160 / 1,452 个候选特征
已检查 192 / 1,452 个候选特征
已检查 224 / 1,452 个候选特征
已检查 256 / 1,452 个候选特征
已检查 288 / 1,452 个候选特征
已检查 320 / 1,452 个候选特征
已检查 352 / 1,452 个候选特征
已检查 384 / 1,452 个候选特征
已检查 416 / 1,452 个候选特征
已检查 448 / 1,452 个候选特征
已检查 480 / 1,452 个候选特征
已检查 512 / 1,452 个候选特征
已检查 544 / 1,452 个候选特征
已检查 576 / 1,452 个候选特征
已检查 608 / 1,452 个候选特征
已检查 640 / 1,452 个候选特征
已检查 672 / 1,452 个候选特征
已检查 704 / 1,452 个候选特征
已检查 736 / 1,452 个候选特征
已检查 768 / 1,452 个候选特征
已检查 800 / 1,452 个候选特征
已检查 832 / 1,452 个候选特征
已检查 864 / 1,452 个候选特征
已检查 896 / 1,452 个候选特征
已检查 928 / 1,452 个候选特征
已检查 960 / 1,452 个候选特征
已检查 992 / 1,452 个候选特征
已检查 1,024 / 1,452 个候选特征
已检查 1,056 / 1,452 个候选特征
已检查 1,088 / 1,452 个候选特征
已检查 1,120 / 1,452 个候选特征
已检查 1,152 / 1,452 个候选特征
已检查 1,184 / 1,452 个候选特征
已检查 1,216 / 1,452 个候选特征
已检查 1,248 / 1,452 个候选特征
已检查 1,280 / 1,452 个候选特征
已检查 1,312 / 1,452 个候选特征
已检查 1,344 / 1,452 个候选特征
已检查 1,376 / 1,452 个候选特征
已检查 1,408 / 1,452 个候选特征
已检查 1,440

,column,valid_count,missing_count,missing_rate,non_missing_nunique,is_all_missing,is_constant
1113,person2_person1_employment_tenure_days_min,0,1282082,1.0,0,True,False
1115,person2_person1_employment_tenure_days_max,0,1282082,1.0,0,True,False
1114,person2_person1_employment_tenure_days_mean,0,1282082,1.0,0,True,False
1065,person_related3_isreference_387L,0,1282082,1.0,0,True,False
1067,person_related3_mainoccupationinc_384A,0,1282082,1.0,0,True,False
165,assignmentdate_4955616D,0,1282082,1.0,0,True,False
1055,person_related2_age_years,0,1282082,1.0,0,True,False
1056,person_related2_employment_years,0,1282082,1.0,0,True,False
1052,person_related2_isreference_387L,0,1282082,1.0,0,True,False
1054,person_related2_mainoccupationinc_384A,0,1282082,1.0,0,True,False


## 6. 时间重构：把日期变成“距申请日多久”

原始日期字段直接表示某个日历日期，但对于信用风险模型，更有意义的通常是：

> 某件事距离本次申请发生了多久。

因此将历史日期字段统一转换为：

```text
date_decision - 历史日期
```

例如：

```text
申请日：2024-06-01
历史事件：2024-05-20
→ 12 天
```

转换后的字段命名为：

```text
原字段名__days_before_decision
```

`date_decision` 本身继续保留作为时间元数据；其他原始日期字段在生成相对天数后从模型候选字段中移除。

这一版只完成转换，不在这里进一步判断负天数、异常日期或潜在泄漏。相关审计可以在项目完成基础建模后单独处理。


In [6]:
surviving_date_cols = [
    field.name
    for field in train_schema
    if (
        dtype_family(field.type) == "date"
        and field.name != DECISION_DATE_COL
        and field.name not in drop_cols
    )
]

date_feature_map = {
    column: f"{column}__days_before_decision"
    for column in surviving_date_cols
}

date_feature_report = pd.DataFrame({
    "source_date_column": list(date_feature_map.keys()),
    "derived_numeric_column": list(date_feature_map.values()),
})

date_feature_report.to_csv(
    REPORT_ROOT / "date_feature_map.csv",
    index=False,
)

print(f"需要转换的日期字段：{len(surviving_date_cols):,}")
display(date_feature_report.head(20))


需要转换的日期字段：67


,source_date_column,derived_numeric_column
0,datefirstoffer_1144D,datefirstoffer_1144D__days_before_decision
1,datelastinstal40dpd_247D,datelastinstal40dpd_247D__days_before_decision
2,datelastunpaid_3546854D,datelastunpaid_3546854D__days_before_decision
3,dtlastpmtallstes_4499206D,dtlastpmtallstes_4499206D__days_before_decision
4,firstclxcampaign_1125D,firstclxcampaign_1125D__days_before_decision
5,firstdatedue_489D,firstdatedue_489D__days_before_decision
6,lastactivateddate_801D,lastactivateddate_801D__days_before_decision
7,lastapplicationdate_877D,lastapplicationdate_877D__days_before_decision
8,lastapprdate_640D,lastapprdate_640D__days_before_decision
9,lastdelinqdate_224D,lastdelinqdate_224D__days_before_decision


## 7. 规则落地：统一处理 Train、Validation、OOT 与 Test

到这里，规则已经固定：

1. 删除 Train 中确定的全空列和常量列；
2. 将历史日期转换成相对 `date_decision` 的天数。

之后不会针对 Validation、OOT 或 Test 重新计算删除规则。

处理方式保持一致：

```text
Train        → 使用固定规则
Validation   → 使用固定规则
OOT          → 使用固定规则
Official Test→ 使用固定规则
```

OOT 的 `target` 不放在公开特征文件中，而是单独写入 `sealed_labels`。这样后续 EDA 和调参阶段不会无意中看到最终 OOT 标签。

宽表整体较大，因此这里用 PyArrow 分批读取 Parquet，每一批转成 Pandas DataFrame 完成实际清洗，再分批写回 Parquet。


In [7]:
def clean_pandas_batch(frame):
    frame = frame.copy()

    decision_date = pd.to_datetime(
        frame[DECISION_DATE_COL],
        errors="coerce",
    )

    frame[DECISION_DATE_COL] = decision_date

    for source_column, output_column in date_feature_map.items():
        source_date = pd.to_datetime(
            frame[source_column],
            errors="coerce",
        )

        frame[output_column] = (
            decision_date - source_date
        ).dt.days.astype("Int32")

    columns_to_drop = [
        column
        for column in [*drop_cols, *surviving_date_cols]
        if column in frame.columns
    ]

    return frame.drop(columns=columns_to_drop)


def build_output_schema(input_schema):
    removed = set(drop_cols) | set(surviving_date_cols)

    fields = [
        field
        for field in input_schema
        if field.name not in removed
    ]

    fields.extend([
        pa.field(output_column, pa.int32())
        for output_column in date_feature_map.values()
    ])

    return pa.schema(fields)


PUBLIC_OUTPUT_PATHS = {
    "train": OUTPUT_ROOT / "train_clean.parquet",
    "validation": OUTPUT_ROOT / "validation_clean.parquet",
    "oot_features": OUTPUT_ROOT / "oot_features_clean.parquet",
    "test": OUTPUT_ROOT / "test_clean.parquet",
}

OOT_TARGET_PATH = (
    SEALED_ROOT / "oot_target_sealed.parquet"
)

train_output_schema = build_output_schema(train_schema)
test_output_schema = build_output_schema(test_schema)

feature_output_schema = pa.schema([
    field
    for field in train_output_schema
    if field.name != TARGET_COL
])

oot_target_schema = pa.schema([
    train_schema.field(ID_COL),
    train_schema.field(TARGET_COL),
])

assert schema_signature(
    feature_output_schema
) == schema_signature(test_output_schema)

WRITER_PATHS = {
    **PUBLIC_OUTPUT_PATHS,
    "oot_target_sealed": OOT_TARGET_PATH,
}

WRITER_SCHEMAS = {
    "train": train_output_schema,
    "validation": train_output_schema,
    "oot_features": feature_output_schema,
    "test": test_output_schema,
    "oot_target_sealed": oot_target_schema,
}


def append_parquet(writers, artifact_name, frame):
    if frame.empty:
        return

    output_schema = WRITER_SCHEMAS[artifact_name]

    table = pa.Table.from_pandas(
        frame,
        schema=output_schema,
        preserve_index=False,
        safe=False,
    )

    if artifact_name not in writers:
        writers[artifact_name] = pq.ParquetWriter(
            WRITER_PATHS[artifact_name],
            output_schema,
            compression="zstd",
            compression_level=3,
        )

    writers[artifact_name].write_table(table)


In [8]:
if SAVE_OUTPUTS:
    writers = {}

    try:
        train_parquet = pq.ParquetFile(TRAIN_PATH)

        for batch_number, record_batch in enumerate(
            train_parquet.iter_batches(
                batch_size=WRITE_ROW_BATCH_SIZE,
                use_threads=True,
            ),
            start=1,
        ):
            frame = clean_pandas_batch(
                record_batch.to_pandas()
            )

            train_mask = (
                frame[WEEK_COL] < VALID_START_WEEK
            )

            validation_mask = (
                (frame[WEEK_COL] >= VALID_START_WEEK)
                & (frame[WEEK_COL] < OOT_START_WEEK)
            )

            oot_mask = (
                frame[WEEK_COL] >= OOT_START_WEEK
            )

            append_parquet(
                writers,
                "train",
                frame.loc[train_mask],
            )

            append_parquet(
                writers,
                "validation",
                frame.loc[validation_mask],
            )

            append_parquet(
                writers,
                "oot_features",
                frame.loc[oot_mask].drop(
                    columns=[TARGET_COL]
                ),
            )

            append_parquet(
                writers,
                "oot_target_sealed",
                frame.loc[
                    oot_mask,
                    [ID_COL, TARGET_COL],
                ],
            )

            del frame, record_batch
            gc.collect()

            if batch_number % 20 == 0:
                processed = (
                    batch_number * WRITE_ROW_BATCH_SIZE
                )
                print(
                    f"Train 已处理约 {processed:,} 行"
                )

        test_parquet = pq.ParquetFile(TEST_PATH)

        for record_batch in test_parquet.iter_batches(
            batch_size=WRITE_ROW_BATCH_SIZE,
            use_threads=True,
        ):
            frame = clean_pandas_batch(
                record_batch.to_pandas()
            )

            append_parquet(
                writers,
                "test",
                frame,
            )

            del frame, record_batch
            gc.collect()

    finally:
        for writer in writers.values():
            writer.close()

    missing_outputs = [
        name
        for name in WRITER_PATHS
        if not WRITER_PATHS[name].exists()
    ]

    assert not missing_outputs, (
        f"以下输出没有生成：{missing_outputs}"
    )

else:
    print("SAVE_OUTPUTS=False：未写出数据产物")


Train 已处理约 200,000 行
Train 已处理约 400,000 行
Train 已处理约 600,000 行
Train 已处理约 800,000 行
Train 已处理约 1,000,000 行
Train 已处理约 1,200,000 行
Train 已处理约 1,400,000 行


## 8. 校验输出结果

清洗后的数据需要满足几个基本条件：

- Train、Validation、OOT 的行数与原时间切分一致；
- 每个数据集中的 `case_id` 仍然唯一；
- Train 与 Validation 字段完全一致；
- OOT features 和 Test 与 Train 去掉 `target` 后的字段一致；
- 被删除的原始字段不再出现；
- 新生成的相对日期字段已经存在；
- OOT features 与单独封存的 OOT target 使用相同的 `case_id` 顺序。

这一步只验证数据结构，不计算任何 OOT 模型指标。


In [9]:
assert SAVE_OUTPUTS, "请先生成输出文件"


output_schemas = {
    name: parquet_schema(path)
    for name, path in PUBLIC_OUTPUT_PATHS.items()
}

assert schema_signature(
    output_schemas["validation"]
) == schema_signature(output_schemas["train"])


train_clean_without_target = pa.schema([
    field
    for field in output_schemas["train"]
    if field.name != TARGET_COL
])

assert schema_signature(
    train_clean_without_target
) == schema_signature(
    output_schemas["oot_features"]
)

assert schema_signature(
    train_clean_without_target
) == schema_signature(
    output_schemas["test"]
)


validation_rows = []

validation_artifacts = {
    "train": PUBLIC_OUTPUT_PATHS["train"],
    "validation": PUBLIC_OUTPUT_PATHS["validation"],
    "oot": PUBLIC_OUTPUT_PATHS["oot_features"],
}

for split_name, path in validation_artifacts.items():
    check = pd.read_parquet(
        path,
        columns=[ID_COL, WEEK_COL],
        engine="pyarrow",
    )

    validation_rows.append({
        "split": split_name,
        "row_count": len(check),
        "unique_case_count": check[ID_COL].nunique(),
        "column_count": len(parquet_schema(path)),
        "week_count": check[WEEK_COL].nunique(),
        "week_min": int(check[WEEK_COL].min()),
        "week_max": int(check[WEEK_COL].max()),
    })

    del check
    gc.collect()


output_validation = pd.DataFrame(validation_rows)

assert output_validation["row_count"].eq(
    output_validation["unique_case_count"]
).all()


for split_name in ["train", "validation", "oot"]:
    expected_rows = int(
        split_summary.loc[
            split_summary["split"].eq(split_name),
            "row_count",
        ].iloc[0]
    )

    actual_rows = int(
        output_validation.loc[
            output_validation["split"].eq(split_name),
            "row_count",
        ].iloc[0]
    )

    assert actual_rows == expected_rows, (
        f"{split_name} 行数不守恒"
    )


oot_feature_ids = pd.read_parquet(
    PUBLIC_OUTPUT_PATHS["oot_features"],
    columns=[ID_COL],
    engine="pyarrow",
)[ID_COL]

sealed_oot_ids = pd.read_parquet(
    OOT_TARGET_PATH,
    columns=[ID_COL],
    engine="pyarrow",
)[ID_COL]

assert oot_feature_ids.reset_index(drop=True).equals(
    sealed_oot_ids.reset_index(drop=True)
), "OOT features 与 OOT target 的 case_id 不一致"

del oot_feature_ids, sealed_oot_ids
gc.collect()


output_validation.to_csv(
    REPORT_ROOT / "output_validation.csv",
    index=False,
)

display(output_validation)
print("输出校验通过")


,split,row_count,unique_case_count,column_count,week_count,week_min,week_max
0,train,1282082,1282082,1214,64,0,63
1,validation,92870,92870,1214,14,64,77
2,oot,151707,151707,1213,14,78,91


输出校验通过


## 9. 保存字段目录和清洗配置

最后保存两份后续 Notebook 会继续使用的文件：

### `feature_catalog.csv`

记录每个字段的：

- 字段名；
- 数据类型；
- 字段角色；
- 字段来源。

### `preprocess_config.json`

记录本轮已经固定的项目规则，包括：

- 时间边界；
- 删除字段；
- 日期转换关系；
- 输入和输出路径。

这两份文件的作用是让 Notebook 6 及之后的步骤直接读取本轮结果，而不需要重新推断清洗规则。


In [10]:
final_schema = output_schemas["train"]
derived_date_cols = set(date_feature_map.values())

final_catalog_rows = []

for position, field in enumerate(
    final_schema,
    start=1,
):
    column = field.name

    if column == ID_COL:
        role = "identifier"
    elif column == TARGET_COL:
        role = "target"
    elif column in TIME_METADATA_COLS:
        role = "time_metadata"
    else:
        role = "model_feature"

    if column in derived_date_cols:
        origin = "derived_from_date"
    else:
        origin = "original"

    final_catalog_rows.append({
        "position": position,
        "column": column,
        "dtype": str(field.type),
        "dtype_family": dtype_family(field.type),
        "role": role,
        "origin": origin,
    })


final_catalog = pd.DataFrame(
    final_catalog_rows
)

FEATURE_CATALOG_PATH = (
    OUTPUT_ROOT / "feature_catalog.csv"
)

final_catalog.to_csv(
    FEATURE_CATALOG_PATH,
    index=False,
)


PREPROCESS_CONFIG_PATH = (
    OUTPUT_ROOT / "preprocess_config.json"
)

run_config = {
    "version": "pandas_simplified_v1",
    "input_train": str(TRAIN_PATH),
    "input_test": str(TEST_PATH),
    "week_column": WEEK_COL,
    "decision_date_column": DECISION_DATE_COL,
    "valid_start_week": VALID_START_WEEK,
    "oot_start_week": OOT_START_WEEK,
    "train_week_range": [
        EXPECTED_WEEK_MIN,
        VALID_START_WEEK - 1,
    ],
    "validation_week_range": [
        VALID_START_WEEK,
        OOT_START_WEEK - 1,
    ],
    "oot_week_range": [
        OOT_START_WEEK,
        EXPECTED_WEEK_MAX,
    ],
    "target_inspected_before_split": False,
    "cleaning_rules_learned_from_train_only": True,
    "automatic_drop_columns": drop_cols,
    "date_feature_map": date_feature_map,
    "split_mapping": str(SPLIT_MAPPING_PATH),
    "feature_catalog": str(FEATURE_CATALOG_PATH),
    "public_outputs": {
        name: str(path)
        for name, path in PUBLIC_OUTPUT_PATHS.items()
    },
    "sealed_oot_target": str(OOT_TARGET_PATH),
}

with PREPROCESS_CONFIG_PATH.open(
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        run_config,
        file,
        ensure_ascii=False,
        indent=2,
        default=str,
    )


display(
    final_catalog
    .groupby(
        ["role", "dtype_family"],
        dropna=False,
    )
    .size()
    .rename("column_count")
    .reset_index()
    .sort_values(
        ["role", "column_count"],
        ascending=[True, False],
    )
)

print("feature_catalog:", FEATURE_CATALOG_PATH)
print("preprocess_config:", PREPROCESS_CONFIG_PATH)
print("Notebook 5 完成")


,role,dtype_family,column_count
0,identifier,numeric,1
3,model_feature,numeric,1150
2,model_feature,categorical,53
1,model_feature,boolean,6
4,target,numeric,1
6,time_metadata,numeric,2
5,time_metadata,date,1


feature_catalog: /kaggle/working/clean_split/feature_catalog.csv
preprocess_config: /kaggle/working/clean_split/preprocess_config.json
Notebook 5 完成
